# 09 · Validation Against Published Experiments

**Covers:** reading numbers from papers, setting the model to them, and building the theory vs simulation vs experiment comparison.
**Experiments:** Hong, Ou and Mandel, *Phys. Rev. Lett.* 59, 2044 (1987), and Kues et al., *Nature* 546, 622 (2017).

Two different questions are asked here:

- **Simulation vs theory:** is the code correct? (all earlier notebooks)
- **Simulation vs experiment:** is the *model* right? When it is not, the gap points to physics the model leaves out.

**Source note:** the Kues et al. numbers below are taken from the conference summary [Reimer et al., FiO 2017](https://eprints.gla.ac.uk/155151/1/155151.pdf), which reports the Nature paper's results. Assumptions are labelled in the code.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

## 1. Hong–Ou–Mandel 1987: what a 100 fs dip says about the photons

The paper reports a time-interval distribution about 100 fs wide. Reading that as the full width at half maximum of the dip in the notebook 06 model, $P(\tau) = \tfrac12(1 - e^{-\tau^2/2\sigma^2})$, gives the photon's spectral bandwidth.

**Gap:** the 1987 dip *depth* is not used here, and the 700 nm wavelength is an illustrative assumption, not a number from the paper.

In [2]:
fwhm_fs = 100.0                                            # "about 100 fs", read as the dip FWHM
sigma = fwhm_fs / (2 * np.sqrt(2 * np.log(2)))
print(f"dip sigma = {sigma:.1f} fs")

def P(tau):
    return 0.5 * (1 - np.exp(-tau**2 / (2 * sigma**2)))
check("P at half the FWHM is half the background", P(fwhm_fs / 2), 0.25)

d_omega = 1 / (np.sqrt(2) * sigma * 1e-15)                 # rms angular bandwidth
dnu = 2 * np.sqrt(2 * np.log(2)) * d_omega / (2 * np.pi)   # FWHM in Hz
lam = 700e-9                                               # ASSUMED wavelength
print(f"photon bandwidth: {dnu / 1e12:.2f} THz, about {lam**2 * dnu / 2.998e8 * 1e9:.1f} nm at 700 nm")

dip sigma = 42.5 fs
PASS  P at half the FWHM is half the background: simulation = 0.2500, theory = 0.2500
photon bandwidth: 6.24 THz, about 10.2 nm at 700 nm


## 2. Single photons vs weak laser light: the 50 % limit

The literature quotes a maximum HOM visibility of ½ for weak coherent states. The same four-mode model as notebook 06 reproduces it: a laser sometimes sends two photons into the *same* input, and those never interfere.

$$V = 1 - \frac{C_{\text{identical}}}{C_{\text{distinguishable}}}$$

In [3]:
N = 8
def op(k):
    ops = [qt.qeye(N)] * 4
    ops[k] = qt.destroy(N)
    return qt.tensor(ops)

aH, aV, bH, bV = [op(k) for k in range(4)]
G = (aH.dag() * bH - aH * bH.dag()) + (aV.dag() * bV - aV * bV.dag())
U = (np.pi / 4 * G).expm()
Na = aH.dag() * aH + aV.dag() * aV
Nb = bH.dag() * bH + bV.dag() * bV
vac, one = qt.basis(N, 0), qt.basis(N, 1)

def coincidence(state):
    return qt.expect(Na * Nb, U * state)

V_photons = 1 - coincidence(qt.tensor(one, vac, one, vac)) / coincidence(qt.tensor(one, vac, vac, one))
check("HOM visibility, single photons", V_photons, 1.0)

alpha = 0.5
phis = np.linspace(0, 2 * np.pi, 16, endpoint=False)      # average over the random relative phase
def coh(phi):
    return qt.coherent(N, alpha * np.exp(1j * phi))
C_id = np.mean([coincidence(qt.tensor(qt.coherent(N, alpha), vac, coh(p), vac)) for p in phis])
C_di = np.mean([coincidence(qt.tensor(qt.coherent(N, alpha), vac, vac, coh(p))) for p in phis])
check("HOM visibility, weak laser light", 1 - C_id / C_di, 0.5)

PASS  HOM visibility, single photons: simulation = 1.0000, theory = 1.0000


PASS  HOM visibility, weak laser light: simulation = 0.5000, theory = 0.5000


## 3. Kues et al. 2017: effective dimension from $g^{(2)}$

One arm of a comb with $K$ equally weighted line pairs is $K$ independent thermal modes, so its heralding-free $g^{(2)} = 1 + 1/K$. The paper reports an effective dimensionality of **10.45 ± 0.53** from such a measurement, and concludes a Schmidt number of 10.

In [4]:
Nth, nbar = 6, 0.02
def g2_multimode(K):
    rho = qt.tensor([qt.thermal_dm(Nth, nbar)] * K)
    Ntot = sum(qt.tensor([qt.num(Nth) if j == k else qt.qeye(Nth) for j in range(K)]) for k in range(K))
    m = qt.expect(Ntot, rho)
    return qt.expect(Ntot * (Ntot - 1), rho) / m**2

for K in [1, 2, 3, 4]:
    check(f"g2 of {K} thermal modes", g2_multimode(K), 1 + 1 / K)

K_meas, K_err = 10.45, 0.53
print(f"implied g2 = {1 + 1 / K_meas:.4f} (range {1 + 1 / (K_meas + K_err):.4f} to {1 + 1 / (K_meas - K_err):.4f})")
print(f"ideal K = 10 lies {(K_meas - 10) / K_err:.2f} error bars from the measurement")

PASS  g2 of 1 thermal modes: simulation = 2.0000, theory = 2.0000
PASS  g2 of 2 thermal modes: simulation = 1.5000, theory = 1.5000
PASS  g2 of 3 thermal modes: simulation = 1.3333, theory = 1.3333
PASS  g2 of 4 thermal modes: simulation = 1.2500, theory = 1.2500
implied g2 = 1.0957 (range 1.0911 to 1.1008)
ideal K = 10 lies 0.85 error bars from the measurement


## 4. Kues et al. 2017: visibility vs tomography fidelity

Reported raw two-photon interference visibilities and tomography fidelities for D = 2, 3, 4. Simple model: the ideal maximally entangled state mixed with white noise, with the noise level set by the measured visibility. Its fidelity is $V + (1 - V)/D^2$.

In [5]:
data = {2: (0.837, 0.885), 3: (0.866, 0.809), 4: (0.864, 0.766)}   # (visibility, fidelity)
rows = []
for D, (V, F_meas) in data.items():
    Phi = sum(qt.tensor(qt.basis(D, k), qt.basis(D, k)) for k in range(D)).unit()
    rho = V * qt.ket2dm(Phi) + (1 - V) * qt.tensor(qt.qeye(D), qt.qeye(D)) / D**2
    F_model = qt.expect(qt.ket2dm(Phi), rho).real
    check(f"D = {D}: model fidelity = V + (1-V)/D^2", F_model, V + (1 - V) / D**2)
    rows.append((D, V, F_model, F_meas, F_meas - F_model))

print("\n D | visibility | model fidelity | measured | measured - model")
for D, V, Fm, Fx, gap in rows:
    print(f" {D} |   {V:.3f}    |     {Fm:.3f}      |  {Fx:.3f}   |   {gap:+.3f}")

# can unequal comb-line amplitudes explain the D = 4 gap?
p = np.array([0.4, 0.3, 0.2, 0.1])
psi = sum(np.sqrt(p[k]) * qt.tensor(qt.basis(4, k), qt.basis(4, k)) for k in range(4))
Phi4 = sum(qt.tensor(qt.basis(4, k), qt.basis(4, k)) for k in range(4)).unit()
print(f"\nfidelity of a strongly unbalanced D = 4 state: {abs(Phi4.dag() * psi) ** 2:.3f}")

PASS  D = 2: model fidelity = V + (1-V)/D^2: simulation = 0.8777, theory = 0.8778
PASS  D = 3: model fidelity = V + (1-V)/D^2: simulation = 0.8809, theory = 0.8809
PASS  D = 4: model fidelity = V + (1-V)/D^2: simulation = 0.8725, theory = 0.8725

 D | visibility | model fidelity | measured | measured - model
 2 |   0.837    |     0.878      |  0.885   |   +0.007
 3 |   0.866    |     0.881      |  0.809   |   -0.072
 4 |   0.864    |     0.873      |  0.766   |   -0.107

fidelity of a strongly unbalanced D = 4 state: 0.944


## Comparison table

| Quantity | Theory / model | Simulation | Experiment | Verdict |
|---|---|---|---|---|
| HOM visibility, weak coherent light | 0.5 | 0.5 | limit quoted in the literature | agrees |
| 1987 dip width → bandwidth | 6.24 THz (Gaussian model) | — | about 100 fs dip | consistent, wavelength assumed |
| Comb effective dimension | $g^{(2)} = 1 + 1/K$ | matches for K = 1–4 | 10.45 ± 0.53 | ideal K = 10 within 0.85 error bars |
| Fidelity, D = 2 | 0.878 | 0.878 | 0.885 | agrees within 0.007 |
| Fidelity, D = 3 | 0.881 | 0.881 | 0.809 | model too high by 0.072 |
| Fidelity, D = 4 | 0.873 | 0.873 | 0.766 | model too high by 0.107 |

**Discussion.** The white-noise model works for qubits but over-predicts fidelity for D = 3 and 4. Unequal line amplitudes alone cannot explain it (a strongly unbalanced state still has fidelity 0.944). Remaining candidates are phase errors in the frequency-bin control, non-white noise, or reconstruction errors in the tomography. These are hypotheses to check against the full paper, not findings.

## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [6]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

10 of 10 checks passed
